# Notebook 06 — Figures 11, 12, 13: ML Feature Importance & Decision Trees

**Data needed:** `data/centroid_chars.csv`

**Output:**
- Figure 11a: Entropy-based feature importance heatmap (DT, RF, XGB, LGBM)
- Figure 11b: Value-based feature importance heatmap
- Figure 12: Decision tree for post-2002 subsample
- Figure 13: Decision tree for pre-2002 subsample

In [5]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import seaborn as sns
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.tree._tree import TREE_UNDEFINED
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier; HAS_XGB = True
except ImportError:
    HAS_XGB = False; print("xgboost not installed — skipping XGB")
try:
    from lightgbm import LGBMClassifier; HAS_LGBM = True
except ImportError:
    HAS_LGBM = False; print("lightgbm not installed — skipping LGBM")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)
from utils.data_utils import (load_cluster_panel, load_cluster_ranking,
                               pivot_and_rank, save_figure, CHARS_45)

plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────
BASE = "/ssd1/songjiangliu/shared/asset_clustering"
ENTROPY_FILE = (
    f"{BASE}/Results/characteristic_entropy/"
    "entropy_characteristics_clustering_results_K_50_lambda_1000000.csv"
)

In [6]:
# ── Load cluster ranking + build rank_map ─────────────────────────
ranking   = load_cluster_ranking()
df        = load_cluster_panel(K=50, lam=1_000_000)
cr, rank_map = pivot_and_rank(df, lam=1_000_000, ranking_df=ranking)

print(f"Cluster return matrix: {cr.shape}")
print(f"rank_map sample: raw 5 -> {rank_map.get(5,'?')}  raw 24 -> {rank_map.get(24,'?')}")

Cluster return matrix: (511, 50)
rank_map sample: raw 5 -> L01  raw 24 -> L50


In [10]:
# ── Load pre-computed entropy file + apply official ranking ────────
# The entropy file has columns: year_month, cluster, feature, differential_entropy
# 'cluster' contains RAW cluster IDs (0-49) — apply rank_map to get L01-L50

print(f"Loading entropy file...")
entropy_long = pd.read_csv(ENTROPY_FILE)
print(f"  Shape: {entropy_long.shape}")
print(f"  Columns: {list(entropy_long.columns)}")
print(f"  Sample:\n{entropy_long.head(3).to_string()}")

# Apply official ranking: raw cluster id -> L01-L50
# Keep raw cluster IDs (0-49) as the classification target.
# rank_map is used only for display labels in the decision tree plots.
print(f"Unique raw clusters: {entropy_long['cluster'].nunique()}")
print(f"Unique features    : {entropy_long['feature'].nunique()}")

Loading entropy file...
  Shape: (1187775, 4)
  Columns: ['year_month', 'cluster', 'feature', 'differential_entropy']
  Sample:
  year_month  cluster feature  differential_entropy
0    1977-01        0    A2ME             -0.593287
1    1977-01        0      AC             -0.113846
2    1977-01        0      AT             -0.361431
Unique raw clusters: 50
Unique features    : 45


In [11]:
# ── Pivot entropy long -> wide (year_month × cluster -> 45 features) 
# Matches reference DTree.ipynb Cell 16 exactly
df_wide = (
    entropy_long
    .pivot_table(
        index=["year_month", "cluster"],
        columns="feature",
        values="differential_entropy",
        aggfunc="mean"
    )
    .reset_index()
)

feature_cols = [c for c in df_wide.columns if c not in ["year_month", "cluster"]]
print(f"Wide entropy matrix: {df_wide.shape}")
print(f"Feature cols ({len(feature_cols)}): {feature_cols[:5]}...")
print(f"Rows with any NaN: {df_wide[feature_cols].isna().any(axis=1).sum()}")

# Drop rows where any feature is NaN
df_wide = df_wide.dropna(subset=feature_cols)
print(f"After dropna: {df_wide.shape}")
print(f"\nDate range: {df_wide['year_month'].min()}  ->  {df_wide['year_month'].max()}")
print(f"Unique clusters: {df_wide['cluster'].nunique()}")

Wide entropy matrix: (26367, 47)
Feature cols (45): ['A2ME', 'AC', 'AT', 'ATO', 'B2M']...
Rows with any NaN: 0
After dropna: (26367, 47)

Date range: 1977-01  ->  2020-12
Unique clusters: 50


In [ ]:
# ── Feature importance: train classifiers on entropy features ──────
# Matches reference: X = 45 entropy cols, y = cluster label (L01-L50)

X_all = df_wide[feature_cols]
y_all = df_wide["cluster"]

print(f"Training set: {X_all.shape[0]:,} rows × {X_all.shape[1]} features")
print(f"Classes: {y_all.nunique()} unique cluster labels")

# Train/test split (same as reference: 80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

classifiers = {
    "DT": DecisionTreeClassifier(max_depth=7, random_state=42),
    "RF": RandomForestClassifier(n_estimators=300, max_depth=8,
                                  random_state=42, n_jobs=-1),
}
if HAS_XGB:
    classifiers["XGB"] = XGBClassifier(n_estimators=300, max_depth=5,
                                         random_state=42, verbosity=0,
                                         eval_metric="mlogloss")
if HAS_LGBM:
    classifiers["LGBM"] = LGBMClassifier(n_estimators=300, max_depth=5,
                                           random_state=42, verbose=-1)

rank_df = {}
for name, clf in classifiers.items():
    print(f"  Training {name}...")
    clf.fit(X_train, y_train)
    imp = pd.Series(clf.feature_importances_, index=feature_cols)
    rank_df[name] = imp.rank(ascending=False).astype(int)
    acc = clf.score(X_test, y_test)
    print(f"    Test accuracy: {acc:.3f}")

ranks_ent = pd.DataFrame(rank_df)
print(f"\nFeature importance ranks:\n{ranks_ent.head(10).to_string()}")

Training set: 26,367 rows × 45 features
Classes: 50 unique cluster labels
  Training DT...
    Test accuracy: 0.451
  Training RF...
    Test accuracy: 0.946
  Training XGB...
    Test accuracy: 0.988
  Training LGBM...


In [ ]:
# ── Figure 11: Feature importance heatmap ─────────────────────────
mean_rank = ranks_ent.mean(axis=1).sort_values()
ranks_sorted = ranks_ent.loc[mean_rank.index]

fig11, ax = plt.subplots(figsize=(5, 11))
sns.heatmap(ranks_sorted, ax=ax, cmap="YlOrRd_r", annot=True, fmt="d",
            annot_kws={"size": 6}, linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Rank (1 = most important)"})
ax.set_title("Figure 11: ML Feature Importance (Entropy-Based)",
             fontsize=10, fontweight="bold", pad=8)
ax.set_xlabel("Classifier")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
save_figure(fig11, "Figure11_ML_Importance_Entropy")
plt.show()
print("Figure 11 saved.")